# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [1]:

from datetime import datetime

JOURNAL_FILE = "journal.txt"

def add_entry(filename, text):
    """Append a timestamped journal entry to the file."""
    with open(filename, "a") as file:
        file.write(f"{datetime.now()} - {text}\n")

def read_journal(filename):
    """Read and print all entries. Handle FileNotFoundError."""
    try:
        with open(filename, "r") as file:
            for line in file:
                print(line.strip())
    except FileNotFoundError:
        print("Journal file not found.")

def search_journal(filename, keyword):
    """Return a list of lines containing the keyword (case-insensitive)."""
    try:
        with open(filename, "r") as file:
            return [line.strip() for line in file if keyword.lower() in line.lower()]
    except FileNotFoundError:
        return []

def count_entries(filename):
    """Count and return the number of journal entries."""
    try:
        with open(filename, "r") as file:
            return sum(1 for line in file if line.strip())
    except FileNotFoundError:
        return 0

def run_journal():
    """Menu-driven journal interface."""
    while True:
        print("\n[A]dd [R]ead [S]earch [C]ount [Q]uit")
        choice = input("Choose an option: ").lower()

        if choice == "a":
            text = input("Enter journal entry: ")
            add_entry(JOURNAL_FILE, text)
            print("Entry added!")

        elif choice == "r":
            read_journal(JOURNAL_FILE)

        elif choice == "s":
            keyword = input("Enter keyword: ")
            print(search_journal(JOURNAL_FILE, keyword))

        elif choice == "c":
            print("Total entries:", count_entries(JOURNAL_FILE))

        elif choice == "q":
            print("Goodbye!")
            break

        else:
            print("Invalid choice.")

# --- QUICK TEST (non-interactive) ---
# add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
# add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
# read_journal(JOURNAL_FILE)
# print("Search results:", search_journal(JOURNAL_FILE, "python"))
# print("Total entries:", count_entries(JOURNAL_FILE))

# run_journal()  # Uncomment to test the interactive menu


---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [3]:

import json

TODO_FILE = "tasks.json"

def load_tasks(filename):
    """Load tasks from a JSON file. Return [] if missing or corrupted."""
    try:
        with open(filename, "r") as file:
            return json.load(file)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def save_tasks(filename, tasks):
    """Save tasks list to a JSON file with indent=2."""
    with open(filename, "w") as file:
        json.dump(tasks, file, indent=2)

def add_task(tasks):
    """Prompt for title and priority, append new task."""
    title = input("Enter task title: ").strip()

    if not title:
        print("Title cannot be empty.")
        return False

    priority = input("Enter priority (high/medium/low): ").lower().strip()

    if priority not in ["high", "medium", "low"]:
        print("Invalid priority.")
        return False

    tasks.append({"title": title, "priority": priority, "done": False})
    print("Task added!")
    return True

def list_tasks(tasks):
    """Display all tasks in a formatted table."""
    if not tasks:
        print("No tasks found.")
        return

    print("Index | Status | Title | Priority")
    print("--------------------------------")

    for i, task in enumerate(tasks):
        status = "Done" if task["done"] else "Pending"
        print(f"{i} | {status} | {task['title']} | {task['priority']}")

def mark_done(tasks):
    """Ask for a task index and mark it as done."""
    try:
        index = int(input("Enter task index: "))

        if 0 <= index < len(tasks):
            tasks[index]["done"] = True
            print("Task marked as done!")
            return True
        else:
            print("Invalid index.")
    except ValueError:
        print("Please enter a valid number.")

    return False

def delete_task(tasks):
    """Ask for a task index and remove the task."""
    try:
        index = int(input("Enter task index: "))

        if 0 <= index < len(tasks):
            tasks.pop(index)
            print("Task deleted!")
            return True
        else:
            print("Invalid index.")
    except ValueError:
        print("Please enter a valid number.")

    return False

def run_todo():
    """Main menu loop with auto-save."""
    tasks = load_tasks(TODO_FILE)

    while True:
        print("\n[A]dd [L]ist [D]one [R]emove [Q]uit")
        choice = input("Choose an option: ").lower()

        if choice == "a":
            if add_task(tasks):
                save_tasks(TODO_FILE, tasks)

        elif choice == "l":
            list_tasks(tasks)

        elif choice == "d":
            if mark_done(tasks):
                save_tasks(TODO_FILE, tasks)

        elif choice == "r":
            if delete_task(tasks):
                save_tasks(TODO_FILE, tasks)

        elif choice == "q":
            print("Goodbye!")
            break

        else:
            print("Invalid choice.")


# --- QUICK TEST (non-interactive) ---
# tasks = load_tasks(TODO_FILE)
# tasks.append({"title": "Test task", "priority": "high", "done": False})
# save_tasks(TODO_FILE, tasks)
# tasks = load_tasks(TODO_FILE)
# list_tasks(tasks)

# run_todo()  # Uncomment to test the interactive menu


---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [2]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("✅ expenses.csv created successfully.")

✅ expenses.csv created successfully.


In [4]:

import csv

def load_expenses(filename):
    """Load and validate expenses from CSV. Skip invalid rows."""
    expenses = []

    with open(filename, "r") as file:
        reader = csv.DictReader(file)

        for row in reader:
            try:
                amount = float(row["amount"])

                if not row["category"]:
                    continue

                row["amount"] = amount
                expenses.append(row)

            except (ValueError, TypeError):
                continue

    return expenses


def total_by_category(expenses):
    """Return {category: total_amount} dict."""
    totals = {}

    for expense in expenses:
        category = expense["category"]
        amount = expense["amount"]

        if category in totals:
            totals[category] += amount
        else:
            totals[category] = amount

    return totals


def most_expensive(expenses):
    """Return the expense dict with the highest amount."""
    if not expenses:
        return None

    return max(expenses, key=lambda x: x["amount"])


def print_report(expenses):
    """Print a formatted expense summary report."""
    total = sum(expense["amount"] for expense in expenses)

    print("===== EXPENSE REPORT =====")
    print(f"Total spent: ${total:.2f}")

    print("\nBreakdown by category:")
    totals = total_by_category(expenses)

    for category, amount in totals.items():
        print(f"{category}: ${amount:.2f}")

    print("\nMost expensive item:")
    expensive = most_expensive(expenses)

    if expensive:
        print(f"{expensive['description']} - ${expensive['amount']:.2f}")
    else:
        print("No expenses found.")


# --- TEST ---
expenses = load_expenses("expenses.csv")
print_report(expenses)


===== EXPENSE REPORT =====
Total spent: $510.13

Breakdown by category:
Food: $162.75
Transport: $58.50
Entertainment: $114.99
Education: $72.49
Utilities: $89.00
Health: $12.40

Most expensive item:
Electricity bill - $89.00


---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [5]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

✅ All test files created successfully.


In [7]:

import json
import csv

def validate_json(filename, required_keys):
    """Validate a JSON file's structure and required keys."""
    report = {"valid": 0, "invalid": 0, "errors": []}

    try:
        with open(filename, "r") as file:
            data = json.load(file)

        if not isinstance(data, list):
            report["invalid"] += 1
            report["errors"].append("JSON root must be a list.")
            return report

        for i, item in enumerate(data):
            if not isinstance(item, dict):
                report["invalid"] += 1
                report["errors"].append(f"Item {i}: Not a dictionary.")
                continue

            missing = [key for key in required_keys if key not in item]

            if missing:
                report["invalid"] += 1
                report["errors"].append(f"Item {i}: Missing keys {missing}")
            else:
                report["valid"] += 1

    except FileNotFoundError:
        report["errors"].append("File not found.")
        report["invalid"] += 1

    except json.JSONDecodeError:
        report["errors"].append("Invalid JSON format.")
        report["invalid"] += 1

    return report


def validate_csv(filename, expected_columns):
    """Validate a CSV file's structure and column count."""
    report = {"valid": 0, "invalid": 0, "errors": []}

    try:
        with open(filename, "r", newline="") as file:
            reader = csv.reader(file)

            header = next(reader, None)

            if header is None:
                report["invalid"] += 1
                report["errors"].append("CSV file is empty.")
                return report

            if len(header) != expected_columns:
                report["invalid"] += 1
                report["errors"].append("Incorrect header column count.")

            for i, row in enumerate(reader, start=2):
                if len(row) == expected_columns:
                    report["valid"] += 1
                else:
                    report["invalid"] += 1
                    report["errors"].append(f"Row {i}: Incorrect column count.")

    except FileNotFoundError:
        report["invalid"] += 1
        report["errors"].append("File not found.")

    return report


def print_validation_report(report, filename):
    """Pretty-print the validation results."""
    print(f"File: {filename}")
    print(f"Valid: {report['valid']}")
    print(f"Invalid: {report['invalid']}")

    if report["errors"]:
        print("Errors:")
        for error in report["errors"]:
            print("-", error)
    else:
        print("No errors found.")


# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")


File: good_data.json
Valid: 3
Invalid: 0
No errors found.
File: bad_data.json
Valid: 2
Invalid: 2
Errors:
- Item 1: Missing keys ['email']
- Item 2: Not a dictionary.
File: corrupted.json
Valid: 0
Invalid: 1
Errors:
- Invalid JSON format.
File: good_data.csv
Valid: 2
Invalid: 0
No errors found.
File: bad_data.csv
Valid: 2
Invalid: 2
Errors:
- Row 3: Incorrect column count.
- Row 4: Incorrect column count.


---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [6]:
import json
import csv

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")

✅ students.json created.


In [8]:

import json
import csv

def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dicts) to a CSV file."""
    try:
        with open(json_filename, "r") as file:
            data = json.load(file)

        if not data:
            print("JSON file is empty.")
            return

        fieldnames = list(data[0].keys())

        with open(csv_filename, "w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)

        print("JSON converted to CSV successfully!")

    except FileNotFoundError:
        print("JSON file not found.")
    except (json.JSONDecodeError, ValueError, TypeError, KeyError):
        print("Invalid JSON data.")


def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dicts)."""
    try:
        with open(csv_filename, "r", newline="") as file:
            reader = csv.DictReader(file)
            data = list(reader)

        with open(json_filename, "w") as file:
            json.dump(data, file, indent=2)

        print("CSV converted to JSON successfully!")

    except FileNotFoundError:
        print("CSV file not found.")
    except (csv.Error, OSError):
        print("Error converting CSV to JSON.")


def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only matching records to CSV."""
    try:
        with open(json_filename, "r") as file:
            data = json.load(file)

        filtered = [
            record for record in data
            if record.get(filter_key) == filter_value
        ]

        if not filtered:
            print("No matching records found.")
            return

        fieldnames = list(filtered[0].keys())

        with open(csv_filename, "w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(filtered)

        print(f"Exported {len(filtered)} matching records!")

    except FileNotFoundError:
        print("JSON file not found.")
    except (json.JSONDecodeError, ValueError, TypeError, AttributeError):
        print("Invalid JSON data.")


# --- TEST ---
# JSON → CSV
json_to_csv("students.json", "students_export.csv")

# CSV → JSON (round-trip test)
csv_to_json("students_export.csv", "students_roundtrip.json")

# Filtered export: only students from Madrid
export_filtered_csv("students.json", "madrid_students.csv", "city", "Madrid")

# Verify round-trip
try:
    with open("students_roundtrip.json", "r") as f:
        roundtrip = json.load(f)

    print(f"\nRound-trip data ({len(roundtrip)} records):")
    for r in roundtrip:
        print(f"  {r}")

except FileNotFoundError:
    print("Round-trip file not found.")


JSON converted to CSV successfully!
CSV converted to JSON successfully!
Exported 2 matching records!

Round-trip data (5 records):
  {'name': 'Alice', 'age': '22', 'city': 'Madrid', 'grade': 'A'}
  {'name': 'Bob', 'age': '25', 'city': 'London', 'grade': 'B'}
  {'name': 'Charlie', 'age': '30', 'city': 'Paris', 'grade': 'A'}
  {'name': 'Diana', 'age': '28', 'city': 'Madrid', 'grade': 'C'}
  {'name': 'Eve', 'age': '23', 'city': 'London', 'grade': 'A'}


---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```

In [9]:

import json
import csv

DB_FILE = "students_db.json"

# --- PERSISTENCE ---

def load_db(filename):
    """Load student records from JSON. Return [] if missing/corrupted."""
    try:
        with open(filename, "r") as file:
            return json.load(file)
    except (FileNotFoundError, json.JSONDecodeError):
        return []


def save_db(filename, students):
    """Save student records to JSON."""
    with open(filename, "w") as file:
        json.dump(students, file, indent=2)


# --- VALIDATION ---

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, error_message)."""
    if not name.strip():
        return False, "Name cannot be empty."

    try:
        if int(age) <= 0 or str(age).strip() != str(int(age)):
            return False, "Age must be a positive integer."
    except (ValueError, TypeError):
        return False, "Age must be a positive integer."

    if "@" not in email:
        return False, "Invalid email."

    if grade.upper() not in ["A", "B", "C", "D", "F"]:
        return False, "Invalid grade."

    return True, ""


# --- CRUD OPERATIONS ---

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    name = input("Enter name: ").strip()
    age = input("Enter age: ").strip()
    email = input("Enter email: ").strip()
    grade = input("Enter grade: ").strip().upper()

    valid, error = validate_student(name, age, email, grade)

    if valid:
        students.append({
            "name": name,
            "age": int(age),
            "email": email,
            "grade": grade
        })
        print("Student added successfully!")
    else:
        print("Error:", error)


def list_students(students):
    """Display all students in a formatted table."""
    if not students:
        print("No students found.")
        return

    print(f"{'Name':<20} {'Age':<8} {'Email':<30} {'Grade':<8}")
    print("-" * 70)

    for student in students:
        print(f"{student['name']:<20} {student['age']:<8} {student['email']:<30} {student['grade']:<8}")


def search_student(students):
    """Search by name (case-insensitive partial match)."""
    term = input("Enter name to search: ").strip().lower()

    results = [
        student for student in students
        if term in student["name"].lower()
    ]

    if results:
        list_students(results)
    else:
        print("No matching students found.")


def update_student(students):
    """Update a student's grade or email by name."""
    name = input("Enter student name: ").strip().lower()

    for student in students:
        if student["name"].lower() == name:
            field = input("Update grade or email? ").strip().lower()

            if field == "grade":
                new_grade = input("Enter new grade: ").strip().upper()

                valid, error = validate_student(
                    student["name"], student["age"],
                    student["email"], new_grade
                )

                if valid:
                    student["grade"] = new_grade
                    print("Grade updated!")
                else:
                    print("Error:", error)

            elif field == "email":
                new_email = input("Enter new email: ").strip()

                valid, error = validate_student(
                    student["name"], student["age"],
                    new_email, student["grade"]
                )

                if valid:
                    student["email"] = new_email
                    print("Email updated!")
                else:
                    print("Error:", error)

            else:
                print("Invalid field.")
            return

    print("Student not found.")


def delete_student(students):
    """Remove a student by name."""
    name = input("Enter student name to delete: ").strip().lower()

    for student in students:
        if student["name"].lower() == name:
            confirm = input("Are you sure? (yes/no): ").lower()

            if confirm == "yes":
                students.remove(student)
                print("Student deleted!")
            else:
                print("Deletion cancelled.")
            return

    print("Student not found.")


# --- IMPORT / EXPORT ---

def import_csv(students, csv_filename):
    """Import students from CSV, validating each row."""
    try:
        with open(csv_filename, "r", newline="") as file:
            reader = csv.DictReader(file)

            for row in reader:
                try:
                    name = row["name"]
                    age = row["age"]
                    email = row["email"]
                    grade = row["grade"]

                    valid, error = validate_student(
                        name, age, email, grade
                    )

                    if valid:
                        students.append({
                            "name": name.strip(),
                            "age": int(age),
                            "email": email.strip(),
                            "grade": grade.strip().upper()
                        })
                    else:
                        print("Skipped row:", row, error)

                except (KeyError, ValueError, TypeError, AttributeError):
                    print("Skipped invalid row:", row)

        print("CSV import completed!")

    except FileNotFoundError:
        print("CSV file not found.")


def export_csv(students, csv_filename):
    """Export all students to a CSV file."""
    if not students:
        print("No students to export.")
        return

    with open(csv_filename, "w", newline="") as file:
        fieldnames = ["name", "age", "email", "grade"]
        writer = csv.DictWriter(file, fieldnames=fieldnames)

        writer.writeheader()
        writer.writerows(students)

    print("Students exported successfully!")


# --- MAIN MENU ---

def run_student_db():
    """Main menu loop."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_student(students)
            save_db(DB_FILE, students)
        elif choice == "l":
            list_students(students)
        elif choice == "s":
            search_student(students)
        elif choice == "u":
            update_student(students)
            save_db(DB_FILE, students)
        elif choice == "d":
            delete_student(students)
            save_db(DB_FILE, students)
        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            import_csv(students, csv_file)
            save_db(DB_FILE, students)
        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)
        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")

# run_student_db()  # Uncomment to test


In [10]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")

import_students.csv created (3 valid, 3 invalid rows).
